# Post-Match Dashboard
Generates a single-figure match report from the per-match parquet files produced
by `store_matches.py` / `migrate_to_per_match.py`.

**Visualisations included**
1. Pass network (home & away)
2. Shot map
3. Momentum / match timeline
4. Key stats bar (PPDA, Field Tilt, Possession, xT proxy)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from mplsoccer import Pitch, VerticalPitch

# ── CONFIG ──────────────────────────────────────────────────────────────
# Point this at any per-match directory created by store_matches / migrate
MATCH_DIR = Path("../data/matches").glob("*/")
MATCH_DIR = sorted(MATCH_DIR)[0]          # pick first match; change as needed
print(f"Loading match: {MATCH_DIR.name}")

In [ ]:
# ── LOAD DATA ───────────────────────────────────────────────────────────
df_meta   = pd.read_parquet(MATCH_DIR / "metadata.parquet")
df_events = pd.read_parquet(MATCH_DIR / "events.parquet")
df_quals  = pd.read_parquet(MATCH_DIR / "qualifiers.parquet")
df_stats  = pd.read_parquet(MATCH_DIR / "stats.parquet")

meta = df_meta.iloc[0]
HOME_ID   = meta["home_team_id"]
AWAY_ID   = meta["away_team_id"]
HOME_NAME = meta["home_team_name"]
AWAY_NAME = meta["away_team_name"]
SCORE     = f"{int(meta['goals_home'])}-{int(meta['goals_away'])}"
DATE      = str(meta["local_date"])[:10]

print(f"{HOME_NAME} {SCORE} {AWAY_NAME}  |  {DATE}")

## Helper: qualifier pivot
Qualifiers are stored in long form. We pivot them wide onto events where needed.

In [ ]:
def attach_qualifiers(events: pd.DataFrame, quals: pd.DataFrame) -> pd.DataFrame:
    """Pivot qualifier_id values wide and merge onto events by event_id."""
    pivot = (
        quals
        .pivot_table(index="event_id", columns="qualifier_id",
                     values="value", aggfunc="first")
        .reset_index()
    )
    pivot.columns.name = None
    pivot.columns = ["event_id"] + [f"q{c}" for c in pivot.columns[1:]]
    return events.merge(pivot, left_on="id", right_on="event_id", how="left")

df = attach_qualifiers(df_events, df_quals)
print(df.shape)

## Subset helpers

In [ ]:
# Event type IDs (from references.py)
TYPE_PASS          = 1
TYPE_SHOT_SAVED    = 15
TYPE_GOAL          = 16
TYPE_MISS          = 13
TYPE_POST          = 14
TYPE_TACKLE        = 7
TYPE_INTERCEPTION  = 8
TYPE_FOUL          = 4

# Qualifier IDs used below
Q_PASS_END_X = 140
Q_PASS_END_Y = 141
Q_BIG_CHANCE = 214

passes_home = df[(df.type_id == TYPE_PASS) & (df.team_id == HOME_ID) & (df.outcome == 1)]
passes_away = df[(df.type_id == TYPE_PASS) & (df.team_id == AWAY_ID) & (df.outcome == 1)]

shots = df[df.type_id.isin([TYPE_SHOT_SAVED, TYPE_GOAL, TYPE_MISS, TYPE_POST])]

print(f"Passes (home successful): {len(passes_home)}")
print(f"Passes (away successful): {len(passes_away)}")
print(f"Shots total: {len(shots)}")

## Pass Networks

In [ ]:
def build_pass_network(passes: pd.DataFrame) -> tuple:
    """
    Returns:
      avg_pos  : DataFrame with player_id, x, y, pass_count
      pairs    : DataFrame with player_a, player_b, count (top-N pairs)
    """
    p = passes.dropna(subset=["player_id", "x", "y"])

    # Average position per player
    avg_pos = (
        p.groupby("player_id")
        .agg(x=("x", "mean"), y=("y", "mean"),
             pass_count=("id", "count"),
             player_name=("player_name", "first"))
        .reset_index()
    )

    # Pass pairs: consecutive passes by same team in same sequence
    # Use qualifier q140/q141 (Pass End X/Y) to identify receiver's position
    # Fallback: use shift-based player pairing within the same period
    p2 = p.sort_values(["period_id", "minute", "second"]).copy()
    p2["next_player"] = p2["player_id"].shift(-1)
    pairs_raw = p2.dropna(subset=["next_player"])
    pairs_raw = pairs_raw[pairs_raw.player_id != pairs_raw.next_player].copy()
    pairs_raw["pair"] = pairs_raw.apply(
        lambda r: tuple(sorted([r.player_id, r.next_player])), axis=1
    )
    pairs = (
        pairs_raw.groupby("pair")["id"].count()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(15)
    )
    pairs[["player_a", "player_b"]] = pd.DataFrame(
        pairs["pair"].tolist(), index=pairs.index
    )
    return avg_pos, pairs

avg_home, pairs_home = build_pass_network(passes_home)
avg_away, pairs_away = build_pass_network(passes_away)

In [ ]:
def draw_pass_network(ax, avg_pos, pairs, pitch, color, title):
    # Scale marker size by pass count
    max_count = avg_pos.pass_count.max()
    sizes = (avg_pos.pass_count / max_count * 900 + 100).values

    pos_dict = avg_pos.set_index("player_id")[["x", "y"]].to_dict("index")

    for _, row in pairs.iterrows():
        a, b = row.player_a, row.player_b
        if a in pos_dict and b in pos_dict:
            lw = row["count"] / pairs["count"].max() * 6
            pitch.lines(pos_dict[a]["x"], pos_dict[a]["y"],
                        pos_dict[b]["x"], pos_dict[b]["y"],
                        ax=ax, lw=lw, color=color, alpha=0.4, zorder=1)

    pitch.scatter(avg_pos.x, avg_pos.y, ax=ax,
                  s=sizes, color=color, edgecolors="white",
                  linewidth=1.5, zorder=2)

    for _, row in avg_pos.iterrows():
        name = str(row.player_name).split()[-1] if pd.notna(row.player_name) else ""
        ax.text(row.x, row.y + 3.5, name, ha="center", va="bottom",
                fontsize=6, color="white", zorder=3)
    ax.set_title(title, color="white", fontsize=10, pad=8)

## Shot Map

In [ ]:
def shot_marker(type_id, outcome):
    """Return (marker, size) for a shot event."""
    if type_id == TYPE_GOAL:
        return "*", 250
    elif type_id == TYPE_SHOT_SAVED:
        return "o", 120
    elif type_id == TYPE_POST:
        return "D", 100
    else:   # miss
        return "x", 90

## Summary Stats: PPDA, Field Tilt, Possession

In [ ]:
def calc_ppda(events: pd.DataFrame, attacking_team_id: str, defending_team_id: str) -> float:
    """
    PPDA = opponent passes in own half / defensive actions.
    attacking_team_id : the team whose passes we count (pressing target).
    defending_team_id : the team doing the pressing.
    Defensive actions = tackles (7) + interceptions (8) + fouls (4) in opponent half (x > 50).
    """
    opp_passes = events[
        (events.team_id == attacking_team_id) &
        (events.type_id == TYPE_PASS) &
        (events.x < 50)   # in the defending team's half
    ]
    def_actions = events[
        (events.team_id == defending_team_id) &
        (events.type_id.isin([TYPE_TACKLE, TYPE_INTERCEPTION, TYPE_FOUL])) &
        (events.x > 50)
    ]
    return len(opp_passes) / max(len(def_actions), 1)


def calc_field_tilt(events: pd.DataFrame, team_id: str) -> float:
    """Field tilt = share of all final-third touches belonging to one team."""
    final_third = events[events.x >= 67]
    if len(final_third) == 0:
        return 0.0
    team_touches = len(final_third[final_third.team_id == team_id])
    return round(team_touches / len(final_third) * 100, 1)


def calc_possession(events: pd.DataFrame, team_id: str) -> float:
    """Simple touch-based possession %."""
    total = len(events[events.team_id.isin([HOME_ID, AWAY_ID])])
    if total == 0:
        return 0.0
    return round(len(events[events.team_id == team_id]) / total * 100, 1)


ppda_home = calc_ppda(df, AWAY_ID, HOME_ID)
ppda_away = calc_ppda(df, HOME_ID, AWAY_ID)
ft_home   = calc_field_tilt(df, HOME_ID)
ft_away   = calc_field_tilt(df, AWAY_ID)
poss_home = calc_possession(df, HOME_ID)
poss_away = calc_possession(df, AWAY_ID)

print(f"PPDA          | {HOME_NAME}: {ppda_home:.2f}  {AWAY_NAME}: {ppda_away:.2f}")
print(f"Field Tilt %  | {HOME_NAME}: {ft_home}%  {AWAY_NAME}: {ft_away}%")
print(f"Possession %  | {HOME_NAME}: {poss_home}%  {AWAY_NAME}: {poss_away}%")

## Match Timeline (momentum)

In [ ]:
def rolling_momentum(events: pd.DataFrame, team_id: str, window: int = 5) -> pd.Series:
    """
    Count events per minute for the team, return rolling-windowed series.
    Positive = home dominance, negative = away.
    """
    e = events[(events.period_id.isin([1, 2])) & events.minute.notna()].copy()
    e["minute"] = e["minute"].astype(int)
    by_min = e.groupby(["minute", "team_id"]).size().unstack(fill_value=0)
    by_min = by_min.reindex(range(1, 95), fill_value=0)

    if HOME_ID not in by_min.columns:
        by_min[HOME_ID] = 0
    if AWAY_ID not in by_min.columns:
        by_min[AWAY_ID] = 0

    momentum = (by_min[HOME_ID] - by_min[AWAY_ID]).rolling(window, center=True).mean()
    return momentum

momentum = rolling_momentum(df, HOME_ID)

## Assemble the Dashboard

In [ ]:
HOME_COL = "#1a78cf"   # blue for home
AWAY_COL = "#e66e2c"   # orange for away
BG       = "#0d1117"

fig = plt.figure(figsize=(20, 14), facecolor=BG)
fig.suptitle(
    f"{HOME_NAME}  {SCORE}  {AWAY_NAME}\n{meta.get('competition_name','')}  |  {DATE}",
    color="white", fontsize=16, fontweight="bold", y=0.98
)

# Grid: 3 rows x 4 cols
gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.3,
                      top=0.92, bottom=0.05, left=0.04, right=0.97)

pitch = Pitch(pitch_type="opta", pitch_color=BG, line_color="#555555",
              linewidth=1, stripe=False)

# ── 1. Pass Network Home ─────────────────────────────────────────────
ax_pn_h = fig.add_subplot(gs[0, 0:2])
ax_pn_h.set_facecolor(BG)
pitch.draw(ax=ax_pn_h)
draw_pass_network(ax_pn_h, avg_home, pairs_home, pitch,
                  HOME_COL, f"{HOME_NAME} — Pass Network")

# ── 2. Pass Network Away ─────────────────────────────────────────────
ax_pn_a = fig.add_subplot(gs[0, 2:4])
ax_pn_a.set_facecolor(BG)
pitch.draw(ax=ax_pn_a)
# Flip away team so they attack left→right from their own perspective
avg_away_flip = avg_away.copy()
avg_away_flip["x"] = 100 - avg_away_flip["x"]
avg_away_flip["y"] = 100 - avg_away_flip["y"]
draw_pass_network(ax_pn_a, avg_away_flip, pairs_away, pitch,
                  AWAY_COL, f"{AWAY_NAME} — Pass Network")

# ── 3. Shot Map ───────────────────────────────────────────────────────
ax_shot = fig.add_subplot(gs[1, 1:3])
ax_shot.set_facecolor(BG)
half_pitch = VerticalPitch(pitch_type="opta", half=True,
                            pitch_color=BG, line_color="#555555")
half_pitch.draw(ax=ax_shot)

for _, s in shots.iterrows():
    col  = HOME_COL if s.team_id == HOME_ID else AWAY_COL
    mkr, sz = shot_marker(s.type_id, s.outcome)
    # VerticalPitch with half=True shows the attacking half;
    # flip away team shots
    sx = s.x if s.team_id == HOME_ID else 100 - s.x
    sy = s.y if s.team_id == HOME_ID else 100 - s.y
    half_pitch.scatter(sx, sy, ax=ax_shot, s=sz, marker=mkr,
                       color=col, edgecolors="white", linewidth=0.6,
                       alpha=0.85, zorder=3)

legend_els = [
    mpatches.Patch(color=HOME_COL, label=HOME_NAME),
    mpatches.Patch(color=AWAY_COL, label=AWAY_NAME),
]
ax_shot.legend(handles=legend_els, loc="lower center", ncol=2,
               facecolor=BG, labelcolor="white", fontsize=8)
ax_shot.set_title("Shot Map", color="white", fontsize=10, pad=8)

# ── 4. Match Timeline ────────────────────────────────────────────────
ax_tl = fig.add_subplot(gs[2, 0:4])
ax_tl.set_facecolor(BG)
mins = momentum.index
ax_tl.fill_between(mins, momentum.clip(lower=0), color=HOME_COL, alpha=0.6)
ax_tl.fill_between(mins, momentum.clip(upper=0), color=AWAY_COL, alpha=0.6)
ax_tl.axhline(0, color="white", linewidth=0.6, linestyle="--")
ax_tl.axvline(45, color="#888888", linewidth=0.8, linestyle=":")

# Mark goals on timeline
goals = df[df.type_id == TYPE_GOAL]
for _, g in goals.iterrows():
    col = HOME_COL if g.team_id == HOME_ID else AWAY_COL
    ax_tl.axvline(g.minute, color=col, linewidth=1.5, alpha=0.9)
    ax_tl.text(g.minute, ax_tl.get_ylim()[1] * 0.9,
               f"⚽{int(g.minute)}'", color=col, fontsize=7, ha="center")

ax_tl.set_xlim(1, 94)
ax_tl.set_xlabel("Minute", color="white", fontsize=8)
ax_tl.set_title("Match Momentum  (↑ home  ↓ away)",
                color="white", fontsize=10)
ax_tl.tick_params(colors="white")
for spine in ax_tl.spines.values():
    spine.set_edgecolor("#444444")

# ── 5. Summary Stats (left panel) ────────────────────────────────────
ax_st = fig.add_subplot(gs[1, 0])
ax_st.set_facecolor(BG)
ax_st.axis("off")

stats_rows = [
    ("Possession %",  f"{poss_home}%",  f"{poss_away}%"),
    ("Field Tilt %",  f"{ft_home}%",    f"{ft_away}%"),
    ("PPDA",          f"{ppda_home:.1f}", f"{ppda_away:.1f}"),
    ("Shots",
     str(len(shots[shots.team_id == HOME_ID])),
     str(len(shots[shots.team_id == AWAY_ID]))),
    ("Goals",
     str(len(goals[goals.team_id == HOME_ID])),
     str(len(goals[goals.team_id == AWAY_ID]))),
]

ax_st.text(0.5, 1.0, "Key Stats", ha="center", va="top",
           color="white", fontsize=10, fontweight="bold",
           transform=ax_st.transAxes)
ax_st.text(0.15, 0.92, HOME_NAME[:12], ha="center", color=HOME_COL,
           fontsize=8, transform=ax_st.transAxes)
ax_st.text(0.85, 0.92, AWAY_NAME[:12], ha="center", color=AWAY_COL,
           fontsize=8, transform=ax_st.transAxes)

for i, (label, vh, va_) in enumerate(stats_rows):
    y = 0.82 - i * 0.16
    ax_st.text(0.5,  y, label, ha="center", color="#aaaaaa",
               fontsize=8, transform=ax_st.transAxes)
    ax_st.text(0.15, y - 0.07, vh, ha="center", color=HOME_COL,
               fontsize=11, fontweight="bold", transform=ax_st.transAxes)
    ax_st.text(0.85, y - 0.07, va_, ha="center", color=AWAY_COL,
               fontsize=11, fontweight="bold", transform=ax_st.transAxes)

# ── 6. Top Players by Passes (right panel) ───────────────────────────
ax_top = fig.add_subplot(gs[1, 3])
ax_top.set_facecolor(BG)
ax_top.axis("off")

if "totalPass" in df_stats.columns:
    top_passers = (
        df_stats.nlargest(8, "totalPass")[["matchName", "totalPass", "team_id"]]
    )
    ax_top.text(0.5, 1.0, "Top Passers", ha="center", va="top",
               color="white", fontsize=10, fontweight="bold",
               transform=ax_top.transAxes)
    for j, (_, row) in enumerate(top_passers.iterrows()):
        col = HOME_COL if row.team_id == HOME_ID else AWAY_COL
        name = str(row.matchName).split()[-1]
        ax_top.text(0.05, 0.88 - j * 0.12, f"{name}",
                    color=col, fontsize=8, transform=ax_top.transAxes)
        ax_top.text(0.80, 0.88 - j * 0.12, str(int(row.totalPass)),
                    color="white", fontsize=8, ha="right",
                    transform=ax_top.transAxes)

plt.savefig(MATCH_DIR / "dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=BG)
plt.show()
print(f"Saved → {MATCH_DIR / 'dashboard.png'}")